# L39 · 综合项目三：端到端后训练管线

**学习目标**
- 把 L31-L35 的后训练各步封装成一条「可复用流水线」
- 理解工业级训练代码的组织方式（配置 → 训练 → 评估 → 产出）
- 用面向对象（L06）搭一个 `PostTrainingPipeline` 类

**前置依赖**：L31（SFT）、L32（RM）、L33（DPO）、L34（PPO）、L35（RLHF）  
**预计时长**：70 分钟  
**技术栈**：`numpy`、`matplotlib`（离线投影版，无需 GPU）

---

## 项目蓝图：像搭积木一样组织后训练

真实后训练工程不是「几个零散脚本」，而是一条**配置驱动**的流水线：

```
config → SFT → (RM) → DPO/PPO → 评估 → 产出模型 + 报告
```
本课我们把前面的算法封装进一个类，跑一次 `pipeline.run()` 就出完整报告。

## 第一步：定义 Pipeline 类（整合各阶段）

In [ ]:
import numpy as np

class PostTrainingPipeline:
    def __init__(self, method="dpo", seed=0):
        self.method = method
        self.seed = seed
        self.report = {}

    def sft(self):
        np.random.seed(self.seed)
        X = np.random.randn(50,4); Wt = np.array([[1,0,0,0],[0,1,0,0]]).T
        Y = X@Wt + np.random.normal(0,0.1,(50,2))
        W = np.random.randn(4,2)*0.1
        for _ in range(200):
            p = X@W; W -= 0.05*(2*X.T@(p-Y)/len(X))
        self.W = W
        return float(((X@W-Y)**2).mean())

    def train_reward(self):
        good = np.random.randn(60,2)+1; bad = np.random.randn(60,2)-1
        rw = np.random.randn(2)*0.1
        for _ in range(300):
            g = rw@good.mean(0)-rw@bad.mean(0)
            rw += 0.1*(1/(1+np.exp(-g))-1)*(good.mean(0)-bad.mean(0))
        self.rw = rw
        return float(rw@good.mean(0)-rw@bad.mean(0))

    def align(self):
        ref = self.W.copy(); sft_loss = self.sft()
        if self.method == "dpo":
            good = np.random.randn(60,2)+1; bad = np.random.randn(60,2)-1
            for _ in range(300):
                ag = self.W@good.mean(0)-ref@good.mean(0)
                ab = self.W@bad.mean(0)-ref@bad.mean(0)
                p = 1/(1+np.exp(-0.5*(ag-ab)))
                self.W -= 0.05*(p-1)*(good.mean(0)-bad.mean(0))
            gap = float(self.rw@(self.W@good.mean(0))-self.rw@(self.W@bad.mean(0)))
            return gap
        return 0.0

    def run(self):
        self.report["sft_loss"] = self.sft()
        self.report["reward_gap"] = self.train_reward()
        self.report["aligned_gap"] = self.align()
        return self.report

print("✅ PostTrainingPipeline 类定义完成")

# 🎯 AHA 顿悟单元格：一行命令跑完整个后训练

运行下面代码。你只需 `pipeline.run()` 一行，就能**自动跑完 SFT→奖励模型→DPO 对齐**，
并吐出一份结构化训练报告：SFT 损失、奖励差距、对齐后的偏好差距。改 `method` 体验不同对齐算法。

> 你刚把 L31-L35 五节课熔成了一条可复用的工业级流水线。在 SOTA 公司，研究员每天就是调 `config`、跑 `pipeline`、看报告。

In [ ]:
# ===== 运行我！一行跑完后训练全流程 =====
for method in ["dpo"]:
    pipe = PostTrainingPipeline(method=method, seed=0)
    report = pipe.run()
    print(f"  🔧 后训练流水线（方法={method}）运行完毕\n")
    print(f"  ├─ SFT 损失：{report['sft_loss']:.3f}  （越低=指令格式越准）")
    print(f"  ├─ 奖励模型偏好差距：{report['reward_gap']:.3f}  （>0=评委已学会分辨）")
    print(f"  └─ 对齐后偏好差距：{report['aligned_gap']:.3f}  （越高=模型越听话）")
    print("\n  📦 产出：已对齐模型权重 W + 训练报告 → 可直接进入 L30 MLOps 发布！")
    print("  ✨ 五节课的算法，如今一行 run() 全跑完。")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课定位**：综合项目三，整合 L31-L35 成可复用 Pipeline（OOP 实践）。  
**易错点**：各阶段随机种子需一致以保证可复现；`align` 中 DPO 分支依赖 `self.rw`（须先 train_reward）；`run()` 顺序固定。  
**AHA 机制**：一行 `run()` 出完整报告，强「工程化封装」专业感，阶段七中段。  
**衔接**：L40 求职（此 Pipeline 是简历亮点）；接 L30 MLOps 发布。  
**真 LLM 升级**：备课笔记指明用 `transformers.Trainer`/`TRL` 替换各 `sft/align` 方法，支持真实模型与数据集，需 GPU。  
**依赖**：`pip install numpy`（`matplotlib` 本课未画图，可省略）。

# 📚 作业 / 下一步

1. 把 `method` 改成不同值（如加 `"ppo"` 分支到 `align`）体验算法切换。
2. 给 Pipeline 加一个 `evaluate()` 方法，输出对齐前后的偏好准确率。
3. 下一课 **L40 求职与作品集：去 SOTA 上班** —— 收官：把 39 节课变成你的求职武器。